In [1]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

In [2]:
def load_yolo_model(model_path):
    model = YOLO(model_path)  # Load the custom YOLO model
    return model

In [3]:
def create_canvas(image, padding_factor=6):
    """
    Create a canvas with the original image in the center and black pixels around it.
    :param image: Input image of the traffic sign.
    :param padding_factor: Factor by which to increase the canvas size.
    :return: Image placed on a larger canvas.
    """
    height, width = image.shape[:2]
    canvas_height = height * padding_factor
    canvas_width = width * padding_factor
    canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8)
    
    # Place the original image in the center of the canvas
    y_offset = (canvas_height - height) // 2
    x_offset = (canvas_width - width) // 2
    canvas[y_offset:y_offset + height, x_offset:x_offset + width] = image
    
    return canvas

In [4]:
def detect_and_crop(image_path, model, conf_threshold=0.25):
    img = cv2.imread(image_path)
    canvas_img = create_canvas(img)  # Create canvas
    results = model(canvas_img)
    crops = []
    for result in results:
        for box, conf in zip(result.boxes.xyxy, result.boxes.conf):
            if conf >= conf_threshold:
                x1, y1, x2, y2 = map(int, box)
                cropped_img = canvas_img[y1:y2, x1:x2].copy()  # Ensure we copy the cropped part
                crops.append(cropped_img)
                return crops  # Save only the first detection
    return crops

In [5]:
def process_images(source_folder, target_folder, model, conf_threshold):
    for folder in range(15):  # Process folders '0' to '14'
        folder_path = os.path.join(source_folder, str(folder))
        target_folder_path = os.path.join(target_folder, str(folder))
        os.makedirs(target_folder_path, exist_ok=True)

        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                cropped_images = detect_and_crop(file_path, model, conf_threshold)
                if not cropped_images:  # No detections
                    target_file_path = os.path.join(target_folder_path, filename)
                    cv2.imwrite(target_file_path, cv2.imread(file_path))  # Copy original image
                else:
                    target_file_path = os.path.join(target_folder_path, filename)
                    cv2.imwrite(target_file_path, cropped_images[0])  # Save only the first detection
                print(f'Processed image saved to {target_file_path}')

In [6]:
# Load the custom YOLO model
model_path = 'sign_detect.pt'
model = load_yolo_model(model_path)

# Configuration for processing
source_folder = './'  # Current working directory
target_folder = './crop'  # Output to a crop directory inside the current working directory
confidence_threshold = 0.25  # Set a low confidence threshold

process_images(source_folder, target_folder, model, confidence_threshold)


0: 640x640 1 traffic-sign, 7.0ms
Speed: 6.0ms preprocess, 7.0ms inference, 379.0ms postprocess per image at shape (1, 3, 640, 640)
Processed image saved to ./crop\0\00001_00000_00000.png

0: 640x640 1 traffic-sign, 8.0ms
Speed: 3.0ms preprocess, 8.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Processed image saved to ./crop\0\00001_00000_00001.png

0: 640x640 1 traffic-sign, 7.0ms
Speed: 3.0ms preprocess, 7.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
Processed image saved to ./crop\0\00001_00000_00002.png

0: 640x640 1 traffic-sign, 7.0ms
Speed: 3.0ms preprocess, 7.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
Processed image saved to ./crop\0\00001_00000_00003.png

0: 640x640 1 traffic-sign, 6.0ms
Speed: 4.0ms preprocess, 6.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
Processed image saved to ./crop\0\00001_00000_00004.png

0: 640x640 1 traffic-sign, 7.0ms
Speed: 3.0ms preprocess, 7.0ms inf